# Buổi 5 — Đánh giá retrieval và mini project

Ta đo TF-IDF, dense retrieval và Reciprocal Rank Fusion trên cùng tập truy vấn có nhãn liên quan.

## Chuẩn bị

~~~bash
python -m pip install -U numpy scikit-learn sentence-transformers
~~~

Trong dự án, gán nhãn query–document trước khi xem kết quả hệ thống.

## 1. Chia tài liệu thành chunk

Chunk nhỏ dễ truy xuất chính xác nhưng có thể mất ngữ cảnh. Overlap giữ lại một phần nội dung giữa hai đoạn liền kề. Hãy luôn lưu source và chỉ số chunk để truy ngược.

In [ ]:
def chunk_words(text, size=12, overlap=3):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Cần size > overlap >= 0")
    words = text.split()
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        if not piece:
            break
        chunks.append({
            "chunk_index": len(chunks),
            "start_word": start,
            "text": " ".join(piece)
        })
    return chunks

sample = (
    "Hệ thống retrieval tìm các đoạn văn liên quan đến câu hỏi. "
    "Chunk nhỏ dễ truy xuất chính xác nhưng có thể thiếu ngữ cảnh. "
    "Overlap giữ lại một phần nội dung giữa hai đoạn liền kề."
)
chunk_words(sample, size=12, overlap=3)

## 2. Corpus và tập query có nhãn

Các ID liên quan là nhãn chuẩn cho ví dụ. Giữ cùng tập query khi so sánh các phương pháp.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ids = [f"d{i:02d}" for i in range(1, 11)]
docs = [
    "TF-IDF đánh trọng số từ dựa trên tần suất trong tài liệu và độ hiếm trong corpus.",
    "IDF giảm ảnh hưởng của từ xuất hiện trong hầu hết tài liệu.",
    "Naive Bayes dùng định lý Bayes để phân loại văn bản như email spam.",
    "Embedding ánh xạ câu thành vector dày, câu cùng ý thường nằm gần nhau.",
    "Semantic search tìm đoạn liên quan dù truy vấn và tài liệu dùng từ khác nhau.",
    "Vector database lưu vector cùng ID, nội dung và metadata để truy vấn lân cận.",
    "Chroma collection lọc bản ghi dựa trên metadata gắn với từng đoạn.",
    "Precision at k đo tỷ lệ kết quả liên quan trong k vị trí đầu.",
    "Recall at k đo phần tài liệu liên quan tìm được trong k kết quả đầu.",
    "MRR ưu tiên hệ thống đưa kết quả liên quan đầu tiên lên thứ hạng cao."
]
queries = [
    {"query": "cách giảm tác động của từ xuất hiện ở mọi nơi", "relevant": {"d01", "d02"}},
    {"query": "mô hình dùng xác suất để nhận diện thư rác", "relevant": {"d03"}},
    {"query": "tìm văn bản có cùng ý nghĩa dù câu hỏi dùng từ khác", "relevant": {"d04", "d05"}},
    {"query": "lưu vector và lọc theo thông tin nguồn", "relevant": {"d06", "d07"}},
    {"query": "đo chất lượng danh sách kết quả tìm kiếm", "relevant": {"d08", "d09", "d10"}}
]
print("Số tài liệu:", len(docs), "| số truy vấn:", len(queries))

## 3. Tạo xếp hạng TF-IDF và embedding

Mã hóa corpus một lần. Mỗi phương pháp trả về ID theo thứ tự liên quan dự đoán.

In [ ]:
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(docs)

def tfidf_ranking(query):
    scores = cosine_similarity(tfidf.transform([query]), doc_tfidf)[0]
    order = np.argsort(scores)[::-1]
    return [ids[i] for i in order]

from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
doc_embeddings = model.encode(
    docs, convert_to_numpy=True, normalize_embeddings=True
)

def dense_ranking(query):
    query_embedding = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    scores = doc_embeddings @ query_embedding
    order = np.argsort(scores)[::-1]
    return [ids[i] for i in order]

rankings = {}
for item in queries:
    q = item["query"]
    rankings[q] = {"tfidf": tfidf_ranking(q), "dense": dense_ranking(q)}
    print("\n", q)
    print("TF-IDF:", rankings[q]["tfidf"][:3])
    print("Dense :", rankings[q]["dense"][:3])

## 4. Precision@k, Recall@k và MRR

Precision@k chia số kết quả liên quan trong top-k cho k. Recall@k chia số kết quả liên quan trong top-k cho tổng số tài liệu liên quan đã gán nhãn. MRR là nghịch đảo thứ hạng của tài liệu liên quan đầu tiên; nếu không có tài liệu liên quan thì điểm bằng 0.

In [ ]:
def retrieval_metrics(ranking, relevant_ids, k=3):
    relevant_ids = set(relevant_ids)
    top_k = ranking[:k]
    hits = sum(doc_id in relevant_ids for doc_id in top_k)
    precision = hits / k
    recall = hits / len(relevant_ids) if relevant_ids else 0.0

    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1.0 / rank
            break

    return {
        "precision_at_k": precision,
        "recall_at_k": recall,
        "mrr": reciprocal_rank
    }

def evaluate(method, k=3):
    rows = [
        retrieval_metrics(rankings[item["query"]][method], item["relevant"], k)
        for item in queries
    ]
    return {
        metric: float(np.mean([row[metric] for row in rows]))
        for metric in rows[0]
    }

print("TF-IDF:", evaluate("tfidf"))
print("Dense :", evaluate("dense"))

## 5. Ghép ranking bằng RRF

RRF cộng 1/(c + rank) cho mỗi tài liệu trong từng danh sách. Rank bắt đầu từ 1. Phương pháp này dùng thứ hạng thay vì yêu cầu điểm TF-IDF và cosine cùng thang đo.

In [ ]:
def reciprocal_rank_fusion(rankings_to_fuse, c=60):
    scores = {}
    for ranking in rankings_to_fuse:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (c + rank)
    return sorted(scores, key=scores.get, reverse=True)

for item in queries:
    q = item["query"]
    rankings[q]["rrf"] = reciprocal_rank_fusion([
        rankings[q]["tfidf"],
        rankings[q]["dense"]
    ])
    print("\n", q, "| RRF top-3:", rankings[q]["rrf"][:3])

print("\nRRF:", evaluate("rrf"))

## Mini project

1. Thay corpus bằng ít nhất 10 đoạn từ tài liệu của bạn; lưu source và chỉ số chunk.
2. Tạo ít nhất 5 query, mỗi query có một hoặc nhiều ID liên quan đã xác định trước.
3. So sánh Precision@3, Recall@3 và MRR của TF-IDF, dense retrieval và RRF.
4. Tìm hai query có lỗi, đọc top-3 và giải thích nguyên nhân.
5. Đề xuất một thay đổi có thể kiểm tra được: đổi chunk size, thêm metadata filter hoặc thử mô hình khác.

Không cần LLM sinh câu trả lời; trọng tâm là chất lượng bước truy xuất.

## Gán nhãn và metric qua một ví dụ

Giả sử ranking là [d3, d1, d2, d4], relevant IDs là {d1, d4}, k=3. Top-3 có một hit: Precision@3=1/3. Có hai đoạn relevant tổng cộng và tìm thấy một: Recall@3=1/2. Hit đầu tiên ở rank 2 nên reciprocal rank=1/2.

Precision lấy k làm mẫu số; Recall lấy số relevant đã gán nhãn làm mẫu số. MRR là trung bình reciprocal rank trên nhiều query, kể cả query không có hit (đóng góp 0).

In [ ]:
example_ranking = ["d3", "d1", "d2", "d4"]
example_relevant = {"d1", "d4"}
example = retrieval_metrics(example_ranking, example_relevant, k=3)
example

## Macro average và query không có đáp án

Tính metric riêng cho từng query rồi lấy trung bình để mỗi nhu cầu người dùng có trọng số bằng nhau. Hãy xem từng hàng trước khi nhìn trung bình; cùng một điểm aggregate có thể che một query thất bại hoàn toàn.

Query không có relevant IDs không có Recall xác định. Tách chúng ra để đánh giá no-answer: hệ thống có thể nhận biết điểm thấp và từ chối đưa ra kết quả không? Đừng gán Recall bằng 1 để làm đẹp điểm.

In [ ]:
# So sánh macro average theo từng query cho ba ranking
for method in ["tfidf", "dense", "rrf"]:
    per_query = []
    for item in queries:
        ranking = rankings[item["query"]][method]
        per_query.append(
            retrieval_metrics(ranking, item["relevant"], k=3)
        )
    macro = {
        key: float(np.mean([row[key] for row in per_query]))
        for key in per_query[0]
    }
    print(method, macro)

## Tính RRF bằng tay

Nếu lexical ranking là [d1,d2,d3], dense ranking là [d3,d1,d4], c=60 thì d1 nhận 1/61 + 1/62 ≈ 0.03252; d3 nhận 1/63 + 1/61 ≈ 0.03227. Vì d1 đứng hạng cao ở cả hai danh sách nên nhỉnh hơn d3.

RRF cộng theo rank, không cộng raw TF-IDF score với cosine score. Nếu hai hệ thống cùng ưu tiên một tài liệu sai, RRF vẫn có thể giữ tài liệu đó ở vị trí cao; luôn đánh giá ranking hợp nhất.

In [ ]:
lexical_demo = ["d1", "d2", "d3"]
dense_demo = ["d3", "d1", "d4"]
fused_demo = reciprocal_rank_fusion([lexical_demo, dense_demo], c=60)
print(fused_demo)
print("d1:", round(1/61 + 1/62, 5))
print("d3:", round(1/63 + 1/61, 5))

## Mẫu phân tích lỗi retrieval

Với mỗi query thất bại, lưu query, relevant IDs, top-5 IDs, nội dung đoạn và metadata nguồn. Phân loại lỗi: query không rõ; relevant label thiếu; chunk bị cắt mất ngữ cảnh; keyword không có trong vocabulary; embedding cùng chủ đề nhưng không trả lời; metadata filter loại mất đáp án.

Chỉ đổi một biến trong lần thử tiếp theo. Ví dụ nếu đoạn đúng bị cắt ở ranh giới, thử overlap; nếu mã sản phẩm không khớp, thêm lexical retrieval/hybrid; nếu filter loại đáp án, sửa metadata. Chạy lại cùng các query để biết thay đổi có thực sự giúp không.

## Checklist bàn giao mini project

- Tối thiểu 10 đoạn, mỗi đoạn có ID và source.
- Tối thiểu 5 query, có relevant IDs và tiêu chí relevance.
- Cùng query/corpus/k cho TF-IDF, dense và RRF.
- Bảng P@3, Recall@3, MRR theo phương pháp.
- Hai ví dụ lỗi có nội dung nguồn và giả thuyết nguyên nhân.
- Một thay đổi được thử riêng và kết quả trước/sau.